[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/terrell/terrell.ipynb)

# Terrell Rotation

A body moving at nearly the speed of light is contracted along its motion, yet no photograph shows the contraction. A photograph records the light that reaches the camera at one instant, and light from the far parts of a passing body set out earlier than light from its near parts, when the body was further back. The delay and the contraction together turn the picture instead of squashing it: a passing cube is seen turned, showing the face that trails behind it, and a sphere still shows a circular outline. Terrell and Penrose pointed this out in 1959, more than half a century after the contraction itself.

Events, and the rays of light between them, are modelled as antivectors, directions in spacetime; a ray is null, `ray.squared() == 0`, and the directions of the camera's sky are the rays arriving at it. In the body's own frame nothing moves: there the camera, at `frame << event`, takes an ordinary photograph of a body at rest, each ray running straight from a point of the body to the camera, and the boost `frame >> rays` carries the photograph to the camera's frame. A boost is linear, so rays in one hyperplane through the camera, where `plane & ray == 0`, stay in one, and those rays are a circle on the sky: the boost maps circles on the sky to circles. The outline of a sphere is such a circle in its own frame, so it is one in every frame, and the whole photograph is the one taken at rest from where the camera is in the body's frame: turned, not squashed.

In the vector notation of special relativity, a line of sight at angle $\theta$ from the body's motion in the camera's frame is at $\theta'$ in the body's, with $\cos\theta' = \frac{\cos\theta + \beta}{1 + \beta\cos\theta}$: seen at right angles to its motion, the body is photographed from $\cos\theta' = \beta$, turned by $\arcsin\beta$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import STA
from examples.util.animation import show_animation
from examples.relativity.terrell import render

np.set_printoptions(precision=1, suppress=True)

ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Events, and the rays of light between them, are antivectors: directions in spacetime, intrinsic.
Antivector = ga.gatype.antivector()
Rotor = ga.gatype.rotor()

# One unit along the camera's time, and unit directions of space.
time = -mv.t.dual()                                          # [] Antivector-
space = stack([mv.x, mv.y, mv.z]).dual()                     # [3] Antivector-
# The direction the body travels, the camera's line of sight to its track, and up.
travel, line_of_sight, up = space

## 1. The camera in the body's frame

The camera sits at the origin of its own frame and photographs the body when the light from the body's centre reaches it. That light set out from an event on the camera's past light cone, back along the line of sight as far as the track the body moves along. In the body's frame that event is the body's centre, and the camera is somewhere beside it: `frame << event` reads an event in the body's frame. Seen at right angles to its motion, the body finds the camera behind the perpendicular, by the speed as a fraction of the distance, and that is where its photograph is taken from.

In [ ]:
# The distance from the camera to the track, and the speeds of the cases, as fractions of the speed of light.
distance = 4.5
speeds = np.array([0.0, 0.6, 0.9])

In [ ]:
def boost(speeds: np.ndarray) -> Rotor:
    """The body's frame at each speed, a turn in the plane of the motion and time."""
    return ((travel & time).dual() * (np.arctanh(speeds) / 2)).exp()            # [...] Rotor+


def sighting(distance: float, angles: np.ndarray) -> Antivector:
    """The events where the body's centre sends the light the camera receives along a line of sight at
    each angle from the motion: the ray back along the line of sight, scaled to reach the track."""
    sight = (mv.xy * (-angles / 2)).exp() >> travel                             # [...] Antivector-
    return (sight - time) * (distance / (mv.y & sight))                         # [...] Antivector-


def rest_camera(frame: Rotor, centre: Antivector) -> Antivector:
    """Where the camera is in the body's frame when it receives the light the body's centre sent from
    the event `centre`: the camera's event relative to it, seen from the body, without its time."""
    camera = -(frame << centre)                                                 # [...] Antivector-
    return camera - (mv.t & camera) * time                                      # [...] Antivector-


frame = boost(speeds)                                                           # [cases] Rotor+
camera = rest_camera(frame, sighting(distance, np.pi / 2))                      # [cases] Antivector-
# How far behind the perpendicular the camera is, as a fraction of its distance.
leaning = (mv.x & camera) / camera.squared().square_root()                      # [cases] Scalar+
print("turned by, in degrees:", np.degrees(np.arcsin(-leaning.to_array())))

## 2. Photographs of a passing pair

In the body's frame the photograph is an ordinary one of a body at rest. The camera's sky is drawn on a stereographic map, which draws circles on the sky as circles, and each point of the map is a ray arriving at the camera. `frame << ray` reads that ray in the body's frame, where it runs straight back from where the camera is to the pair at rest, and the first surface it meets there colours the point; the GPU does this for every sample of every photograph. The pair is photographed each time its centre is seen along the same lines of sight at every speed.

In [ ]:
# The sphere's radius and the cube's half-width, and how far above and below the centre of the pair their
# centres are; the angles between the sphere's meridians and between its parallels, and how far its parallels
# reach from its equator.
radius = 1.0
half_width = 1.0
height = 1.6
meridian_spacing = np.pi / 4
parallel_spacing = np.pi / 6
parallel_reach = np.pi / 3
# The lines of sight to the centre of the pair as it passes, as angles from the motion, and the duration
# of a frame.
n_frames = 72
sightings = np.linspace(3 * np.pi / 4, np.pi / 4, n_frames)
duration_ms = 60

Each lane is one speed, from the top: at rest, at six tenths and at nine tenths of the speed of light, on a stereographic map of the camera's sky, which draws circles on the sky as circles. The cube's face ahead is red, the face behind blue, and the face towards the camera violet. At rest the pair shows the face it comes with, then its front, then the face behind as it leaves. At nine tenths of the speed of light it shows the face behind nearly all the way, the sphere's grid turned with it, and the sphere's outline stays round.

In [ ]:
def cameras():
    """Per frame, where the camera is in the body's frame at every speed, when the pair's centre is seen along
    that frame's line of sight."""
    for angle in sightings:
        yield rest_camera(frame, sighting(distance, angle))                    # [cases] Antivector-


sphere = height * up                                                           # [] Antivector-
cube = -height * up                                                            # [] Antivector-
show_animation(render.animate_photographs(frame, cameras(), sphere, radius, cube, half_width, meridian_spacing,
                                          parallel_spacing, parallel_reach), "terrell_rotation", duration_ms)

## 3. The outline, a circle on the sky

The outline of the sphere in the body's frame is where the rays from the camera graze it, a circle on the sphere square to the line to the camera. Carried to the camera's frame, `frame >> rays`, those rays lie in one hyperplane through the camera, so the join of any four of them, each scaled to one unit of the camera's time, vanishes: the outline is a circle on the sky at every speed.

In [ ]:
# The rays around the outline.
n_outline = 96

In [ ]:
def rays(camera: Antivector, points: Antivector) -> Antivector:
    """The rays from points of a body at rest to the camera at rest beside it, in their frame."""
    offsets = camera - points                                                   # [...] Antivector-
    # A ray takes as long as it is long.
    return offsets + time * offsets.squared().square_root()                     # [...] Antivector-


def outline(camera: Antivector, centre: Antivector, radius: float, angles: np.ndarray) -> Antivector:
    """Where the rays from the camera graze a sphere, at each angle around the line to the camera."""
    offset = camera - centre                                                    # [...] Antivector-
    distance = offset.squared().square_root()                                   # [...] Scalar+
    # The grazing points ring the line from the sphere's centre to the camera. The ring is built around the
    # direction to a camera straight in front, and turned onto the camera's, which is never straight behind.
    turn = ((offset / distance) * (-line_of_sight).inverse()).square_root()     # [...] Rotor+
    nearest = (radius / distance) * -line_of_sight + (1 - (radius / distance).squared()).square_root() * up
    circle = (mv.zx * (angles / 2)).exp() >> nearest[..., None]                 # [..., angles] Antivector-
    return centre + radius * (turn[..., None] >> circle)                        # [..., angles] Antivector-


# The outline as the first photograph sees it at every speed, its rays scaled to one unit of the camera's time.
first = next(cameras())                                                         # [cases] Antivector-
grazing = frame[:, None] >> rays(first[:, None], outline(first, sphere, radius, np.linspace(0, 2 * np.pi, n_outline + 1)))
points = grazing / (mv.t & grazing)                                             # [cases, outline + 1] Antivector-
joined = points[:, 0] & points[:, 20] & points[:, 50] & points[:, 70]           # [cases] Scalar+
print("join of four of the outline's rays, at each speed:", joined.to_array())

In [ ]:
# checks
# Seen at right angles to its motion, the body is photographed turned by the arcsine of the speed; the sphere's
# outline is a circle on the sky at every speed.
np.testing.assert_allclose(leaning.to_array(), -speeds, atol=1e-12)
np.testing.assert_allclose(joined.to_array(), 0, atol=1e-12)